# OSC AlphaGenome setup
## imports

In [2]:
from alphagenome_research.model import dna_model
from alphagenome import colab_utils
from alphagenome.data import gene_annotation
from alphagenome.data import genome
from alphagenome.data import transcript
from alphagenome.data import ontology
from alphagenome.interpretation import ism
from alphagenome.models import dna_client
from alphagenome.models import variant_scorers
from alphagenome.visualization import plot_components

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd
import numpy as np
from pysam import VariantFile
from io import StringIO
from tqdm import tqdm
import os
import gc

os.environ['XLA_PYTHON_CLIENT_PREALLOCATE']='true'
os.environ['XLA_PYTHON_CLIENT_MEM_FRACTION'] = '0.9' # pre allocates XX% of total GPU instead of the default 75%

pd.set_option('display.max_columns', None)

## common variables

In [3]:
LMNA_START = 156_082_572
LMNA_END = 156_140_081
gene_symbol = "LMNA"
LMNA_INTERVAL = genome.Interval('chr1', 156_082_572, 156_140_081)


BASE_PATH = '/users/PAS2905/coraalbers/'
AG_DATA_PATH = '/users/PAS2905/coraalbers/ag/ag_data/'

HG38_FASTA_PATH = '/users/PAS2905/coraalbers/ag/hg38.fa'
HG38_GTF_PATH = '/users/PAS2905/coraalbers/ag/ag_data/gencode.v46.annotation.gtf.gz.feather'
HG38_SPLICE_START_PATH = '/users/PAS2905/coraalbers/ag/ag_data/gencode.v46.splice_sites_starts.feather'
HG38_SPLICE_END_PATH = '/users/PAS2905/coraalbers/ag/ag_data/gencode.v46.splice_sites_ends.feather'
CALIBRATION_PATH = f'{AG_DATA_PATH}calibration_scores.pb'

gtf = pd.read_feather( 'https://storage.googleapis.com/alphagenome/reference/gencode/' 'hg38/gencode.v46.annotation.gtf.gz.feather' )

output_modalities = ['atac',	
    'cage',	
    'chip_histone',	
    'chip_tf',	
    'contact_maps',	
    'dnase',	
    'procap',	
    'rna_seq',	
    'splice_junctions',	
    'splice_site_usage',	
    'splice_sites']

requested_outputs = {dna_client.OutputType.ATAC,
        dna_client.OutputType.CAGE,
        dna_client.OutputType.DNASE,
        dna_client.OutputType.PROCAP,
        dna_client.OutputType.RNA_SEQ,
        dna_client.OutputType.SPLICE_SITES,
        dna_client.OutputType.SPLICE_SITE_USAGE,
        dna_client.OutputType.SPLICE_JUNCTIONS,
        dna_client.OutputType.CONTACT_MAPS,
        dna_client.OutputType.CHIP_HISTONE,
        dna_client.OutputType.CHIP_TF}

## model initialization

In [4]:
all_folds_model = dna_model.create_from_huggingface( 'all_folds',
    organism_settings={ 
        dna_model.Organism.HOMO_SAPIENS: dna_model.OrganismSettings( 
            fasta_path=HG38_FASTA_PATH, 
            gtf_feather_path=HG38_GTF_PATH, 
            splice_site_starts_feather_path=HG38_SPLICE_START_PATH, 
            splice_site_ends_feather_path=HG38_SPLICE_END_PATH, 
            calibration_path=CALIBRATION_PATH,
        ), dna_model.Organism.MUS_MUSCULUS: dna_model.OrganismSettings() } 
)

print('all folds model initialized!')

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

all folds model initialized!


In [5]:
df = pd.read_parquet('outputs/ism_reg_var_scores/var_scores_pred_ccre_central_0_Distal enhancer.parquet')

In [7]:
df.head(20)

,variant_id,scored_interval,gene_id,gene_name,gene_type,gene_strand,junction_Start,junction_End,output_type,variant_scorer,track_name,track_strand,Assay title,ontology_curie,biosample_name,biosample_type,biosample_life_stage,data_source,endedness,genetically_modified,transcription_factor,histone_mark,gtex_tissue,raw_score
142,chr1:156338016:A>C,chr1:155587039-156635615:.,None,None,None,NaN,None,None,ATAC,"CenterMaskScorer(requested_output=ATAC, width=...",UBERON:0002084 ATAC-seq,.,ATAC-seq,UBERON:0002084,heart left ventricle,tissue,adult,encode,paired,False,None,None,None,-0.016740
452,chr1:156338016:A>C,chr1:155587039-156635615:.,None,None,None,NaN,None,None,DNASE,"CenterMaskScorer(requested_output=DNASE, width...",UBERON:0002084 DNase-seq,.,DNase-seq,UBERON:0002084,heart left ventricle,tissue,adult,encode,paired,False,None,None,None,-0.001025
2056,chr1:156338016:A>C,chr1:155587039-156635615:.,None,None,None,NaN,None,None,CHIP_TF,"CenterMaskScorer(requested_output=CHIP_TF, wid...",UBERON:0002084 TF ChIP-seq CTCF,.,TF ChIP-seq,UBERON:0002084,heart left ventricle,tissue,adult,encode,single,False,CTCF,None,None,-0.010229
2057,chr1:156338016:A>C,chr1:155587039-156635615:.,None,None,None,NaN,None,None,CHIP_TF,"CenterMaskScorer(requested_output=CHIP_TF, wid...",UBERON:0002084 TF ChIP-seq POLR2A,.,TF ChIP-seq,UBERON:0002084,heart left ventricle,tissue,adult,encode,paired,False,POLR2A,None,None,-0.006223
3049,chr1:156338016:A>C,chr1:155587039-156635615:.,None,None,None,NaN,None,None,CHIP_HISTONE,CenterMaskScorer(requested_output=CHIP_HISTONE...,UBERON:0002084 Histone ChIP-seq H3K27ac,.,Histone ChIP-seq,UBERON:0002084,heart left ventricle,tissue,adult,encode,single,False,None,H3K27ac,None,-0.012278
3050,chr1:156338016:A>C,chr1:155587039-156635615:.,None,None,None,NaN,None,None,CHIP_HISTONE,CenterMaskScorer(requested_output=CHIP_HISTONE...,UBERON:0002084 Histone ChIP-seq H3K27me3,.,Histone ChIP-seq,UBERON:0002084,heart left ventricle,tissue,adult,encode,single,False,None,H3K27me3,None,0.002964
3051,chr1:156338016:A>C,chr1:155587039-156635615:.,None,None,None,NaN,None,None,CHIP_HISTONE,CenterMaskScorer(requested_output=CHIP_HISTONE...,UBERON:0002084 Histone ChIP-seq H3K36me3,.,Histone ChIP-seq,UBERON:0002084,heart left ventricle,tissue,adult,encode,single,False,None,H3K36me3,None,0.000336
3052,chr1:156338016:A>C,chr1:155587039-156635615:.,None,None,None,NaN,None,None,CHIP_HISTONE,CenterMaskScorer(requested_output=CHIP_HISTONE...,UBERON:0002084 Histone ChIP-seq H3K4me1,.,Histone ChIP-seq,UBERON:0002084,heart left ventricle,tissue,adult,encode,single,False,None,H3K4me1,None,0.001157
3053,chr1:156338016:A>C,chr1:155587039-156635615:.,None,None,None,NaN,None,None,CHIP_HISTONE,CenterMaskScorer(requested_output=CHIP_HISTONE...,UBERON:0002084 Histone ChIP-seq H3K4me3,.,Histone ChIP-seq,UBERON:0002084,heart left ventricle,tissue,adult,encode,single,False,None,H3K4me3,None,-0.000699
3054,chr1:156338016:A>C,chr1:155587039-156635615:.,None,None,None,NaN,None,None,CHIP_HISTONE,CenterMaskScorer(requested_output=CHIP_HISTONE...,UBERON:0002084 Histone ChIP-seq H3K9me3,.,Histone ChIP-seq,UBERON:0002084,heart left ventricle,tissue,adult,encode,single,False,None,H3K9me3,None,-0.002205
